# Ice Temperature Profiles

* Description: The product “internal ice-sheet temperature” is composed by maps of internal temperature for the regions in which the Robin model is valid D5-2 [RD.9] along with the relative uncertainty estimation and a map of the quality of the estimation.
* Original Data Source: https://zenodo.org/records/8052519
* Reference: https://www.science.org/doi/10.1126/sciadv.adi0186 
* OSC entry: https://opensciencedata.esa.int/products/ice-temperature-profiles-antarctica/collection
* License: CC-BY-4.0 
* Repo Folder: ./datasets/ice_temp

In [1]:
import xarray as xr
import cartopy.crs as ccrs
import hvplot.xarray

from pystac_client import Client
from IPython.display import display

catalog = Client.open("https://eoapi.workspace.earthcode.eox.at/stac/")
item = next(catalog.search(collections=["ice-temperature-profiles-antarctica"], ids=["SM_TEST_MIR_ITUDP4_20130101T000000_20141231T000000_200_001_0"]).items())

item

<Item id=SM_TEST_MIR_ITUDP4_20130101T000000_20141231T000000_200_001_0>

In [2]:
item.assets["data"].href

'https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/polar_cube_datasets/ice_sheet_temperature/SM_TEST_MIR_ITUDP4_20130101T000000_20141231T000000_200_001_0.zarr/'

In [3]:
ds = xr.open_zarr(item.assets["data"].href)
ds

<xarray.Dataset> Size: 17MB
Dimensions:      (x: 200, y: 224, depth: 91)
Coordinates:
  * depth        (depth) int16 182B 0 50 100 150 200 ... 4350 4400 4450 4500
Dimensions without coordinates: x, y
Data variables:
    latitude     (x, y) float64 358kB dask.array<chunksize=(200, 224), meta=np.ndarray>
    longitude    (x, y) float64 358kB dask.array<chunksize=(200, 224), meta=np.ndarray>
    QualityFlag  (x, y) float32 179kB dask.array<chunksize=(200, 224), meta=np.ndarray>
    spatial_ref  int64 8B ...
    Tice         (x, y, depth) float32 16MB dask.array<chunksize=(200, 224, 15), meta=np.ndarray>
Attributes: (12/21)
    product:               Internal ice sheet temperature
    product_version:       2.0
    institution:           IFAC-CNR, IGE
    description:           developed from SMOS L3 (CATDS)
    source:                IFAC L4Tice processor
    project:               ESA CryoSMOS project (contract n.4000112262/14/I-NB)
    ...                    ...
    geospatial_lat_max:    0
    geospatial_lon_min:    -180
    geospatial_lon_max:    180
    spatial_resolution:    25 km
    srid:                  EPSG:6932
    creation_date:         2019-09-20T15:14:23

In [4]:
lon = ((ds["longitude"] + 180) % 360) - 180

tice = ds["Tice"].assign_coords(longitude=lon, latitude=ds["latitude"])


tice = tice.rename("Tice (Kelvin)")

tice.hvplot.quadmesh(
    x="longitude",
    y="latitude",
    groupby="depth",
    geo=True,
    crs=ccrs.PlateCarree(),
    projection=ccrs.SouthPolarStereo(central_longitude=0, true_scale_latitude=-71),
    coastline=True,
    cmap="coolwarm",
    colorbar=True,
    clabel=tice.name,
    width=850,
    height=700,
    title="Ice temperature by depth",
)

BokehModel(combine_events=True, render_bundle={'docs_json': {'0e91f14c-0936-4842-800f-888cb1ea8da9': {'version…